In [0]:
-- Step 1: Read files into a table as binary
CREATE OR REPLACE TABLE ai2605.ai.pdf_raw_data AS
SELECT * FROM read_files('/Volumes/ai2605/ai/unstructured/pdfs/', format => 'binaryFile');

-- Step 2: Parse the PDF content
CREATE OR REPLACE TABLE ai2605.ai.parsed_documents AS
SELECT 
    path, 
    ai_parse_document(content) AS parsed_json
FROM ai2605.ai.pdf_raw_data;

-- Step 3: Extract content (e.g., pulling out elements)

SELECT path, element.type, element.content 
FROM ai2605.ai.parsed_documents
LATERAL VIEW EXPLODE(parsed_json:document:elements::array<struct<bbox:array<struct<coord:array<bigint>,page_id:bigint>>,confidence:decimal(4,4),content:string,description:string,id:bigint,type:string>>) AS element;

-- COMMAND ----------

ALTER TABLE ai2605.ai.parsed_documents SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 30 days');
ALTER TABLE ai2605.ai.parsed_documents SET TBLPROPERTIES (delta.enableChangeDataFeed = true);


-- Index creation failed
-- Invalid embedding source column type in schema. Field 'content' has type binary. Supported field types are string
-- select * from ai2605.ai.pdf_raw_data;

-- COMMAND ----------

CREATE OR REPLACE TABLE ai2605.ai.parsed_documents AS
SELECT 
    path, 
    CAST(ai_parse_document(content) AS STRING) AS parsed_json
FROM ai2605.ai.pdf_raw_data;

select * from ai2605.ai.parsed_documents;